# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FARNHELL/ML-INTERNSHIP/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

# Lane: Refresh / Content Opportunity Scoring

**Task type:** Scoring (a ranking task under the hood). The deliverable an editor actually uses is not a single yes/no label — it's an ordered refresh queue: "start at the top, work down until you run out of hours this week." That output shape is a score, not a class.

Underneath the score sits a binary classification model (probability that a page is declining). I turn that probability into a rank by sorting pages high-to-low. So the honest description is: classification model, scoring framing, ranking use. This matches the framing-skill's mapping — "Which ones first?" → ranking/scoring — because the decision from Week 1 was explicitly which pages first, not is this page declining, yes/no. A bare classification label alone wouldn't tell an editor with time for 40 pages this week which 40 to pick out of the 16,262 flagged as declining; a score does.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**What I'd predict:** is_declining_label — 1 if a page's trend_direction == "down", else 0 .

Where it comes from — and why that matters: this is a defined rule, not an observed later-window outcome. trend_direction == "down" itself comes from trend_pct < -20%, which compares impressions_last_30d against impressions_prev_30d — two windows that both already sit inside the same 90-day export. There is no future window in this starter CSV; the "decline" already happened by the time the row was exported. That's the framing skill's warning in practice ("a label that comes from someone's rule means your model learns the rule, not the world") — so I am not claiming this model predicts what a page will do next month. It's a proxy for "is this page currently in an observed downswing," and I'll say so in every claim in weeks 6–7, not just here.

The honest fix for that gap already exists in the data-dictionary path: the warehouse release (fact_content_daily_performance, weeks 3+) has real forward time, so a later week can redefine the target as an observed future decline over a held-out window per client, which is the correct target for this lane. For this week, is_declining_label is the best available proxy, and trend_direction / trend_pct are permanently excluded as features (the label-trap gotcha) so the model isn't just re-deriving its own label.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric:** precision@K on the ranked queue, where K is however many pages an editor can actually refresh in a cycle (Week 1 framed this as "limited hours per week"). Concretely: take the model's score, sort descending, take the top K, and ask what share of those K are truly declining (by the proxy label)? That's the number an editor can act on directly — it answers "if I trust this queue and work top-down, how often am I not wasting my hour?"

**Why not accuracy:** the label is roughly balanced (54.2% / 45.8%, shown below), so accuracy is hard to game but also doesn't map to the actual decision — an editor never classifies all 30,000 pages, they work a short list.

**Secondary / training metric:** ROC-AUC. Since the queue is built by sorting a predicted probability, AUC is the right metric while comparing candidate models (it's threshold-free and rewards correct ordering, not just a single cutoff). I'll report both: AUC to justify which model earns a place in the pipeline, precision@K to say what that means for the editor's actual week.

What 'good' looks like: better than the 54.2% base rate at every K, and clearly better than a simple fixed-rule baseline — both of which I check for real below.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
import pandas as pd

csv_url = "https://raw.githubusercontent.com/FARNHELL/ML-INTERNSHIP/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(csv_url)

# Build the proxy target the same way scripts/01_prepare_features.py does.
# trend_direction / trend_pct are the label's own source columns, so they are
# never allowed as model features (the "label trap" from skills/flyrank/flyrank-data).
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")
print("One row = one pseudonymized content item (page), aggregated over its trailing 90-day window.")
print(f"Distinct clients in this slice: {df['client_id'].nunique()}")

lane_cols = [
    "content_id", "client_id", "content_type", "word_count", "days_since_last_update",
    "impressions_90d", "ctr", "avg_position", "engagement_rate", "ai_traffic_pct",
    "trend_direction", "is_declining_label",
]
df[lane_cols].head(8)

Rows: 30,000   Columns: 45
One row = one pseudonymized content item (page), aggregated over its trailing 90-day window.
Distinct clients in this slice: 32


,content_id,client_id,content_type,word_count,days_since_last_update,impressions_90d,ctr,avg_position,engagement_rate,ai_traffic_pct,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3221.0,20,3803,0.76,10.6,5.88,0.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,2481.0,25,15320,0.05,20.3,0.00,0.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,3515.0,20,12581,0.09,36.5,0.00,0.0,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,NaN,22,11751,0.49,6.2,1.28,0.0,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,2803.0,14,19140,0.13,44.0,0.00,0.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3080.0,20,3970,0.03,8.5,0.00,0.0,down,1
6,content_9a34b442b552,client_8722616204,keyword article,3059.0,20,20,0.00,7.0,0.00,0.0,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,NaN,22,1724,0.06,21.2,3.57,0.0,stable,0


In [5]:
# Sketch of the target column: how it's distributed, and against the base rate
# an editor is implicitly comparing any model to ("just guess everyone is declining").
label_counts = df["is_declining_label"].value_counts().sort_index()
label_pct = (df["is_declining_label"].value_counts(normalize=True) * 100).round(1).sort_index()

print("is_declining_label counts:")
print(label_counts.rename(index={0: "0 (not declining)", 1: "1 (declining)"}))
print()
print("is_declining_label share:")
print(label_pct.rename(index={0: "0 (not declining)", 1: "1 (declining)"}).astype(str) + "%")
print()
print(f"Base rate an editor is implicitly working against: {label_pct[1]}% of the catalog is flagged declining —")
print("too large to hand-triage, and close enough to a coin flip that a lucky guess isn't a real baseline either.")

is_declining_label counts:
is_declining_label
0 (not declining)    13738
1 (declining)        16262
Name: count, dtype: int64

is_declining_label share:
is_declining_label
0 (not declining)    45.8%
1 (declining)        54.2%
Name: proportion, dtype: object

Base rate an editor is implicitly working against: 54.2% of the catalog is flagged declining —
too large to hand-triage, and close enough to a coin flip that a lucky guess isn't a real baseline either.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [6]:
# Test the most obvious fixed rule an editor might write by hand:
# "flag a page as declining if it's stale (not updated in 6+ months) OR ranking poorly (avg_position > 20)"
from sklearn.metrics import precision_score, recall_score

candidates = ["avg_position", "ctr", "word_count", "days_since_last_update",
              "engagement_rate", "ai_traffic_pct", "search_volume", "content_age_days"]

print("How strongly does any single signal line up with the label, alone?")
for c in candidates:
    corr = df[c].fillna(0).corr(df["is_declining_label"])
    print(f"  {c:25s} corr with label = {corr:+.3f}")

rule_flag = ((df["days_since_last_update"] > 180) | (df["avg_position"] > 20)).astype(int)
base_rate = df["is_declining_label"].mean()

print()
print(f"Base rate (flag nothing, or flag everything as 'declining'): {base_rate:.3f} precision by definition")
print(f"Fixed-rule precision (stale OR poor position): {precision_score(df['is_declining_label'], rule_flag):.3f}")
print(f"Fixed-rule recall:    {recall_score(df['is_declining_label'], rule_flag):.3f}")
print(f"Fixed-rule flags {rule_flag.mean():.1%} of the catalog")

How strongly does any single signal line up with the label, alone?
  avg_position              corr with label = -0.029
  ctr                       corr with label = -0.062
  word_count                corr with label = +0.119
  days_since_last_update    corr with label = +0.081
  engagement_rate           corr with label = -0.013
  ai_traffic_pct            corr with label = +0.002
  search_volume             corr with label = -0.014
  content_age_days          corr with label = -0.164

Base rate (flag nothing, or flag everything as 'declining'): 0.542 precision by definition
Fixed-rule precision (stale OR poor position): 0.527
Fixed-rule recall:    0.282
Fixed-rule flags 29.0% of the catalog


# **What the numbers say**:
 every individual signal correlates weakly with the label — the strongest, content_age_days, is still only ±0.16. There's no single column, and no obvious pair, that cleanly separates decliners from the rest.

That shows up directly when I write the most intuitive rule I can (stale content OR poor position): it flags 29% of the catalog, and its precision (0.527) is actually below the base rate (0.542) — the rule does slightly worse than guessing every page is declining, while also missing 72% of the pages that truly are (recall 0.282). Two reasonable-sounding thresholds, combined by hand, don't beat noise.

That's the gap ML fills: the real signal isn't in any one column, it's in how many weak, differently-shaped signals (freshness, position, content type, keyword competition, AI-traffic share, engagement) move together for a given page. That's exactly the pattern a model can weigh and combine and an if-statement can't hand-tune — which lines up with what the Week 1 starter pipeline already showed on this same data: a simple baseline scorer landed around AUC 0.63, while a Random Forest combining the same signals reached AUC 0.75. The lift comes from combining weak signals, not from finding one strong one.

What this doesn't prove yet: this is a snapshot correlation check, not a validated model — Week 5–6 is where I actually train, hold out by client_id, and check for leakage before trusting any of this for real.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.